Import Statements

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
from typing import Dict, List, Union, Final

Configuration and Inputs

In [ ]:
# Ticker for HDFC Bank on NSE ( that is just the name to be given to the yfinance lib to fethc the stock data)
TICKER: Final[str] = "HDFCBANK.NS"

# Timeframe as per project guidelines
START_DATE: Final[str] = "2025-03-04"
END_DATE: Final[str] = "2026-03-04"

# Using 91-day T-Bill yield as the Risk-Free Rate (r) for Section B
# Historically approx 6.75% for this period
RISK_FREE_RATE: Final[float] = 0.0675 

# Specific Expiry Dates for HDFC Bank Futures
EXPIRIES: Final[Dict[str, datetime]] = {
    "Feb_2026": datetime(2026, 2, 26),
    "Mar_2026": datetime(2026, 3, 26),
    "Feb_2027": datetime(2027, 2, 4)
}

Actual Data Collection 

In [ ]:
def get_stock_data(ticker: str, start: str, end: str) -> pd.DataFrame:
    """Fetches historical closing prices and returns a cleaned DataFrame."""
    stock_df: pd.DataFrame = yf.download(ticker, start=start, end=end)
    # Extract only the Close price column
    return stock_df[['Close']].copy()

# Execute data fetch
df_raw: pd.DataFrame = get_stock_data(TICKER, START_DATE, END_DATE)

Theoretical Pricing Function

In [ ]:
def calculate_theoretical_price(spot: float, r: float, expiry: datetime, current_date: datetime) -> float:
    """
    Computes the theoretical futures price using continuous compounding.
    Type-safe implementation ensuring spot and r are floats.
    """
    t_days: int = (expiry - current_date).days
    
    # If the contract has already expired, the price converges to spot
    if t_days <= 0:
        return float(spot)
    
    t_years: float = t_days / 365.0
    return float(spot * np.exp(r * t_years))

Daily Calculation Processing

In [ ]:
# List to store our structured results
pricing_results: List[Dict[str, Union[datetime, float]]] = []

for date, row in df_raw.iterrows():
    # Type extraction from yfinance multi-index or standard series
    current_spot: float = float(row['Close'].iloc[0] if hasattr(row['Close'], 'iloc') else row['Close'])
    current_dt: datetime = date.to_pydatetime()

    # Initializing the row entry
    row_entry: Dict[str, Union[datetime, float]] = {
        "Date": current_dt,
        "Spot_Price": current_spot
    }

    # Calculate theoretical prices for each expiry contract
    for label, expiry_dt in EXPIRIES.items():
        row_entry[f"Theoretical_{label}"] = calculate_theoretical_price(
            current_spot, RISK_FREE_RATE, expiry_dt, current_dt
        )

    pricing_results.append(row_entry)

# Convert results into a structured DataFrame
df_pricing: pd.DataFrame = pd.DataFrame(pricing_results)

Export to Csv using Pandas

In [ ]:
# Save to CSV for project submission
output_filename: str = "HDFC_SectionB_Pricing_Analysis.csv"
df_pricing.to_csv(output_filename, index=False)

print(f"Success: {output_filename} has been generated.")
df_pricing.head() # Preview the first few rows